In [ ]:
from dataloaders.edge_dataEdgeClassificationDataLoader import EdgeClassificationDataLoader
from models.EdgeClassifier import RFClassifier, XGBClassifier
import pandas as pd
from dataloaders.GNNDataLoader import GNNDataLoader
from data_processing.config import DataProcessingConfig
import os

In [ ]:
# Scenarios
case_study = 'manhattan_case_study'
results_dir = os.path.join('..', 'studies', case_study, 'results')
scenario_names = [
    'test_manhattan_scenario_1',
    'test_manhattan_scenario_2',
    'test_manhattan_scenario_3',
]

scenarios = [os.path.join(results_dir, sc) for sc in scenario_names]

# Create config with shorter simulation duration for testing
config = DataProcessingConfig(
    sim_start=0,
    sim_end=1800
)

# Set to True only when data needs to be reprocessed
overwrite = False

In [ ]:
loader = GNNDataLoader(scenarios, config, overwrite=overwrite)
data, *masks = loader.load_data()
train_masks, val_masks, test_masks = masks if masks else (None, None, None)

In [ ]:
def get_classifications(classifier_cls, dataloader):
    n_estimators = 100
    classifier = classifier_cls(dataloader, n_estimators=n_estimators)
    return classifier.classify_edges()

In [ ]:
edge_type = 'vr_graph'
load_dir = os.path.join(config.base_data_dir, config.train_dir, '_20250716_163457')
load_dir = None  # Set to None to use default loading behavior
dataloader = EdgeClassificationDataLoader(scenarios, edge_type, config, overwrite, load_dir=load_dir)
y_val_proba_vr, y_test_proba_vr = get_classifications(RFClassifier, dataloader)
# y_val_proba_vr_xgb, y_test_proba_vr_xgb = get_classifications(XGBClassifier, dataloader)

In [ ]:
edge_type = 'rr_graph'
load_dir = os.path.join(config.base_data_dir, config.train_dir, '_20250717_131757')
# load_dir = None  # Set to None to use default loading behavior
dataloader = EdgeClassificationDataLoader(scenarios, edge_type, config, overwrite, load_dir=load_dir)
y_val_proba_rr, y_test_proba_rr = get_classifications(RFClassifier, dataloader)
# y_val_proba_rr_xgb, y_test_proba_rr_xgb = get_classifications(XGBClassifier, dataloader)

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = dataloader.load_data()

In [ ]:
# Hyperparameter tuning for RF and XGB classifiers
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Prepare data for scikit-learn (RF/XGB expect numpy arrays)
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()
X_val_np = X_val.to_numpy()
y_val_np = y_val.to_numpy()

# Random Forest hyperparameter grid
tuned_parameters_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced']
}
rf = RandomForestClassifier(random_state=42)
gs_rf = GridSearchCV(rf, tuned_parameters_rf, cv=3, scoring='f1', n_jobs=-1, verbose=2)
gs_rf.fit(X_train_np, y_train_np)
print('Best RF params:', gs_rf.best_params_)
print('Best RF F1 score:', gs_rf.best_score_)

# XGBoost hyperparameter grid
tuned_parameters_xgb = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
}
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
gs_xgb = GridSearchCV(xgb, tuned_parameters_xgb, cv=3, scoring='f1', n_jobs=-1, verbose=2)
gs_xgb.fit(X_train_np, y_train_np)
print('Best XGB params:', gs_xgb.best_params_)
print('Best XGB F1 score:', gs_xgb.best_score_)


In [ ]:
edge_type = 'rr_graph'
dataloader_rr = EdgeClassificationDataLoader(scenarios, edge_type, config, overwrite)
y_val_proba_rr, y_test_proba_rr = get_classifications(RFClassifier, dataloader)
y_val_proba_rr_xgb, y_test_proba_rr_xgb = get_classifications(XGBClassifier, dataloader)

In [ ]:
df = pd.DataFrame({
    # 'y_val_proba': y_val_proba_vr,
    'y_val_pred': y_val_proba_vr > 0.3,
    'edge_init_label': X_val['edge_init_label'],
    'label': y_val.squeeze()
})

df.value_counts()

In [ ]:
df[df.label == 1].value_counts(normalize=True)

In [ ]:
df_xgb = pd.DataFrame({
    # 'y_val_proba': y_val_proba_vr,
    'y_val_pred': y_val_proba_rr > 0.3,
    'edge_init_label': X_val['edge_init_label'],
    'label': y_val['label']
})

df_xgb.value_counts()

In [ ]:
df_xgb[df_xgb.label == 1].value_counts(normalize=True)

In [ ]:
# Diagnostic cell to understand node_ids and edge_index mapping
import torch

def inspect_graph_node_mapping(graph_data, graph_idx=0, max_examples=5):
    """
    Inspect how node_ids map to PyG's internal indices in the edge_index tensor.
    This helps verify that we're correctly interpreting the edge connections.
    """
    graph = graph_data[graph_idx]
    
    print("=== Graph Structure Analysis ===")
    
    # Print node types and their features
    for node_type in ['request', 'vehicle']:
        if hasattr(graph[node_type], 'x') and hasattr(graph[node_type], 'node_ids'):
            print(f"\n{node_type.capitalize()} Nodes:")
            print(f"- Number of {node_type} nodes: {graph[node_type].x.shape[0]}")
            print(f"- Feature dimensions: {graph[node_type].x.shape[1]}")
            
            # Show a few example node IDs
            ids = graph[node_type].node_ids.numpy()
            print(f"- Sample node_ids: {ids[:max_examples]} {'...' if len(ids) > max_examples else ''}")
    
    # Print edge types and their connections
    edge_type = ('vehicle', 'connects', 'request')
    if hasattr(graph[edge_type], 'edge_index'):
        print(f"\nEdge Type: {edge_type}")
        edge_index = graph[edge_type].edge_index
        print(f"- Number of edges: {edge_index.shape[1]}")
        
        # Show a few example edges with their node IDs
        print("\n=== Example Edge Mappings ===")
        print("Internal Index -> Actual Node ID")
        for i in range(min(max_examples, edge_index.shape[1])):
            src_idx = edge_index[0, i].item()
            dst_idx = edge_index[1, i].item()
            
            # Map internal indices to actual node IDs
            src_id = graph['vehicle'].node_ids[src_idx].item()
            dst_id = graph['request'].node_ids[dst_idx].item()
            
            print(f"Edge {i}: ({src_idx} -> {dst_idx}) maps to Vehicle ID {src_id} -> Request ID {dst_id}")
        
        # Verify that indices in edge_index are within bounds
        max_vehicle_idx = graph['vehicle'].x.shape[0] - 1
        max_request_idx = graph['request'].x.shape[0] - 1
        
        valid_vehicle_indices = (edge_index[0] >= 0) & (edge_index[0] <= max_vehicle_idx)
        valid_request_indices = (edge_index[1] >= 0) & (edge_index[1] <= max_request_idx)
        
        if not torch.all(valid_vehicle_indices):
            print(f"WARNING: Some vehicle indices in edge_index are out of bounds!")
            invalid_indices = edge_index[0][~valid_vehicle_indices].unique().tolist()
            print(f"Invalid vehicle indices: {invalid_indices}")
        
        if not torch.all(valid_request_indices):
            print(f"WARNING: Some request indices in edge_index are out of bounds!")
            invalid_indices = edge_index[1][~valid_request_indices].unique().tolist()
            print(f"Invalid request indices: {invalid_indices}")
        
        print("\n=== Summary ===")
        print(f"All indices valid: {torch.all(valid_vehicle_indices) and torch.all(valid_request_indices)}")

# Run the inspection function
if data is not None and len(data) > 0:
    inspect_graph_node_mapping(data)
else:
    print("No data available to inspect.")

=== Graph Structure Analysis ===

Request Nodes:
- Number of request nodes: 1
- Feature dimensions: 42
- Sample node_ids: [3179577] 

Vehicle Nodes:
- Number of vehicle nodes: 120
- Feature dimensions: 22
- Sample node_ids: [0 1 2 3 4] ...

Edge Type: ('vehicle', 'connects', 'request')
- Number of edges: 25

=== Example Edge Mappings ===
Internal Index -> Actual Node ID
Edge 0: (2 -> 0) maps to Vehicle ID 2 -> Request ID 3179577
Edge 1: (3 -> 0) maps to Vehicle ID 3 -> Request ID 3179577
Edge 2: (6 -> 0) maps to Vehicle ID 6 -> Request ID 3179577
Edge 3: (10 -> 0) maps to Vehicle ID 10 -> Request ID 3179577
Edge 4: (11 -> 0) maps to Vehicle ID 11 -> Request ID 3179577

=== Summary ===
All indices valid: True


In [ ]:
# Demonstrating how to correctly map predictions back to original node IDs
def get_vehicle_request_assignments(graph, predictions=None, threshold=0.5):
    """
    Maps edge predictions back to original vehicle and request IDs.
    
    Args:
        graph: A heterogeneous graph from your dataset
        predictions: Optional edge prediction tensor (1 = assignment, 0 = no assignment)
                    If None, we'll use the ground truth labels if available
        threshold: Probability threshold for binary predictions (if predictions are continuous)
    
    Returns:
        List of (vehicle_id, request_id) tuples representing assignments
    """
    edge_type = ('vehicle', 'connects', 'request')
    
    # Get edge index
    edge_index = graph[edge_type].edge_index
    
    # Get assignment tensor (either predictions or ground truth)
    if predictions is not None:
        # If predictions are probabilities, convert to binary using threshold
        if predictions.dtype == torch.float:
            assignments = predictions > threshold
        else:
            assignments = predictions
    elif hasattr(graph[edge_type], 'y'):
        # Use ground truth if available
        assignments = graph[edge_type].y == 1
    else:
        print("No predictions or ground truth labels available")
        return []
    
    # Map assignments to vehicle and request IDs
    assigned_pairs = []
    for i, is_assigned in enumerate(assignments):
        if is_assigned:
            # Get internal indices from edge_index
            src_idx = edge_index[0, i].item()
            dst_idx = edge_index[1, i].item()
            
            # Map to original IDs using node_ids
            vehicle_id = graph['vehicle'].node_ids[src_idx].item()
            request_id = graph['request'].node_ids[dst_idx].item()
            
            assigned_pairs.append((vehicle_id, request_id))
    
    return assigned_pairs

# Test the function on the first graph (if data is available)
if data is not None and len(data) > 0:
    # Get assignments based on ground truth
    graph = data[0]
    assignments = get_vehicle_request_assignments(graph)
    
    print("=== Vehicle-Request Assignments (based on ground truth) ===")
    print(f"Found {len(assignments)} assignments")
    
    # Print a few examples
    for i, (vehicle_id, request_id) in enumerate(assignments[:5]):
        print(f"Assignment {i+1}: Vehicle {vehicle_id} → Request {request_id}")
    
    if len(assignments) > 5:
        print("...")
else:
    print("No data available to demonstrate assignments.")

=== Vehicle-Request Assignments (based on ground truth) ===
Found 1 assignments
Assignment 1: Vehicle 11 → Request 3179577


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np


def train_mlp_classifier(X_train, y_train, X_val, y_val, edge_type='vr_graph'):
    # Keep only rows where all features are not NaN
    valid_rows = ~X_train.isna().any(axis=1)

    # Apply to both X and y
    X_train = X_train[valid_rows]
    y_train = y_train[valid_rows]

    # Define categorical features for one-hot encoding
    categorical_features = [
        'src_status', 'src_type', 'tgt_status',
    ] if edge_type == 'vr_graph' else [
        'src_status', 'tgt_status'
    ]

    # Create a column transformer for one-hot encoding
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(), categorical_features)
        ],
        remainder='passthrough'  # Keep other features as they are
    )
    # Create a pipeline with preprocessing and classifier
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
    ])
    pipeline.fit(X_train, y_train)
    # Transform the data
    X_train = pipeline.transform(X_train).astype(np.float32)  # Ensure float32 for PyTorch
    X_val = pipeline.transform(X_val).astype(np.float32)  # Ensure float32 for PyTorch

    # X_train = X_train.to_numpy(dtype=np.float32)  # From your processed DataFrame
    y_train = y_train.to_numpy(dtype=np.int64)

    # X_val = X_val.to_numpy(dtype=np.float32)  # From your processed DataFrame
    y_val = y_val.to_numpy(dtype=np.int64)


    # Optional: scale features (good for MLPs)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # Convert to torch tensors
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_loader = DataLoader(train_dataset, batch_size=64)
    val_loader = DataLoader(val_dataset, batch_size=64)

    # Define your MLP model
    class MLPClassifier(nn.Module):
        def __init__(self, input_dim):
            super(MLPClassifier, self).__init__()
            self.fc1 = nn.Linear(input_dim, 128)
            self.fc2 = nn.Linear(128, 64)
            self.out = nn.Linear(64, 1)  # Binary classification

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = self.out(x)
            return x  # raw logits

    # Initialize model
    input_dim = X_train.shape[1]
    model = MLPClassifier(input_dim)
    pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()])  # Calculate positive weight
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # works directly with logits
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    n_epochs = 50
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)#.squeeze()
            loss = criterion(logits, yb.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb).squeeze()
                preds = (torch.sigmoid(logits) > 0.5).long()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())
        print(classification_report(all_labels, all_preds))

In [ ]:
# train_mlp_classifier(X_train, y_train, X_val, y_val, edge_type='vr_graph')